# JiT-S2-VMamba — d_state = 1 (published VMamba-T mixer)

Our JiT-S2-VMamba baseline uses SS2D with `expand = 1` but `d_state = 16`, the default of VMamba's `SS2D` class. Every **published** VMamba model sets `SSM_D_STATE: 1`: VMamba-T (`vmambav2v_tiny_224`, 30M, 82.6%) uses `ssm_ratio 1.0, d_state 1`, no gate. The VMamba paper moves d_state 16 → 1 in step (f) of §4.3, bundled with other changes (+0.3%, +161 img/s), so its effect alone was never measured. This notebook runs the published setting:

$$\text{SS2D, 4 cross-scan directions:}\quad x_t = \bar A_t x_{t-1} + \bar B_t u_t,\qquad y_t = C_t x_t + D u_t,\qquad x_t \in \mathbb{R}^{d_{inner}\times 1}$$

It is a **config-only** change (`d_state: 1`), with the same `JiTVMamba` code and every conditioning arm **off**. It is the SS2D counterpart of the two JiT-Spatial-Mamba notebooks, which replace "4 scans + CrossMerge" with "1 scan + structure-aware state fusion":

| # | model | mixer | LPU | params | GMACs |
|---|---|---|---|---|---|
| 1 | JiT-VMamba, d_state 1 | SS2D: 4 scans + CrossMerge, expand 1, d_state 1, no gate | – | 30.20M | ≈1.23 |
| 2 | JiT-Spatial, gated | Structure-aware SSM: 1 scan + SASF, expand 1, gate | zero-init | 31.46M | ≈1.30 |
| 3 | JiT-Spatial, no gate | Structure-aware SSM: 1 scan + SASF, expand 1, no gate | zero-init | 29.69M | ≈1.19 |
| ref | JiT-VMamba baseline | SS2D, expand 1, d_state 16 | – | 31.03M | ≈1.42 |

GMACs are analytic estimates (64×64, patch 8, 64 tokens); the verify cell of the Spatial notebooks prints the measured it/s of all of them.

---

JiT-S2 on Tiny-ImageNet-200 (64px, patch 8, 200 classes) via the repo's `run_experiment.py` / `evaluate.py`. Comment out the download cell if you attach a dataset instead, and the eval cell if you only want to train.

## 1. Environment  *(Internet ON)*

In [ ]:
import os, sys

# Pick the stack by the image's Python. torch 2.5.1 has no matching torchvision
# for Python 3.13 (Kaggle's newer image), so 3.13 moves to torch 2.6.0; 3.12
# keeps torch 2.5.1 exactly as every earlier run. mamba-ssm 2.2.4 and
# causal-conv1d 1.5.0.post8 publish cp312 and cp313 wheels for both.
PY = f"cp{sys.version_info.major}{sys.version_info.minor}"
if sys.version_info >= (3, 13):
    TORCH, TVISION, TAUDIO, TTAG = "2.6.0", "0.21.0", "2.6.0", "torch2.6"
else:
    TORCH, TVISION, TAUDIO, TTAG = "2.5.1", "0.20.1", "2.5.1", "torch2.5"
print("python", sys.version.split()[0], "->", PY, "| torch", TORCH)

# 1) Pin torch
!pip install -q torch=={TORCH} torchvision=={TVISION} torchaudio=={TAUDIO} \
    --index-url https://download.pytorch.org/whl/cu124

# 2) Download wheels with explicit destination
CAUSAL = f"causal_conv1d-1.5.0.post8+cu12{TTAG}cxx11abiFALSE-{PY}-{PY}-linux_x86_64.whl"
MAMBA  = f"mamba_ssm-2.2.4+cu12{TTAG}cxx11abiFALSE-{PY}-{PY}-linux_x86_64.whl"

os.system(f"wget -q https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.5.0.post8/{CAUSAL} -O /kaggle/working/{CAUSAL}")
os.system(f"wget -q https://github.com/state-spaces/mamba/releases/download/v2.2.4/{MAMBA} -O /kaggle/working/{MAMBA}")
for w in (CAUSAL, MAMBA):
    p = f"/kaggle/working/{w}"
    assert os.path.exists(p) and os.path.getsize(p) > 0, f"wheel download failed: {w}"

!pip install -q /kaggle/working/{CAUSAL}
!pip install -q /kaggle/working/{MAMBA}

# 3) Patch mamba-ssm
import glob
for path in glob.glob("/usr/local/lib/python*/dist-packages/mamba_ssm/utils/generation.py"):
    with open(path) as f: src = f.read()
    new = src.replace(
        "from transformers.generation import GreedySearchDecoderOnlyOutput, SampleDecoderOnlyOutput, TextStreamer",
        "from transformers.generation import GenerateDecoderOnlyOutput, TextStreamer",
    ).replace(
        "output_cls = GreedySearchDecoderOnlyOutput if top_k == 1 else SampleDecoderOnlyOutput",
        "output_cls = GenerateDecoderOnlyOutput",
    )
    if new != src:
        with open(path, "w") as f: f.write(new)
        print(f"\u2705 Patched {path}")

print(">>> RESTART RUNTIME NOW <<<")

## 2. Repo  *(clone + cd)*

In [ ]:
# Clone the repo. The d_state=1 config and its wiring are on main.
import os
REPO_DIR = "/kaggle/working/thesis_Choustoulakis"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/Rodamanthosch/thesis_Choustoulakis.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
# sanity: the d_state knob reaches the model in both entry points
!grep -q 'd_state=m.get("d_state"' scripts/run_experiment.py && grep -q 'd_state=m.get("d_state"' scripts/evaluate.py \
    && echo "wiring OK (train + eval)" || echo "WIRING MISSING -- pull latest main"

## 3. Verify the model  *(first session only; seconds)*

In [ ]:
# === Verify the model (seconds; first session only -- comment out on resume).
# Builds the config exactly as evaluate.py will (on CPU) and checks it is the
# published VMamba-T mixer: SS2D with d_state 1, expand 1, 4 directions, all
# conditioning arms off. If it FAILS, STOP -- do not train the wrong model.
import yaml
from scripts.evaluate import build_model
cfg = yaml.safe_load(open("configs/tiny_imagenet/jit-s2-vmamba-dstate1.yaml"))
m = build_model(cfg)
mx = m.blocks[0].mixer
n = sum(p.numel() for p in m.parameters())
checks = {
    "SS2D mixer":         type(mx).__name__ == "SS2D",
    "d_state = 1":        mx.d_state == 1 and tuple(mx.A_logs.shape) == (4, 384, 1),
    "expand = 1":         mx.d_inner == 384,
    "K = 4 directions":   mx.K == 4,
    "arms off":           m.in_context_len == 0 and m.state_init == "none" and m.ssc == "none",
    "params 30,202,176":  n == 30_202_176,
}
for k, v in checks.items():
    print(f"{'OK  ' if v else 'FAIL'} {k}")
print(f"{n/1e6:.2f}M params")
assert all(checks.values()), "model check FAILED"
del m

## 4. Download Tiny-ImageNet  *(to /tmp; comment out if you attach a dataset)*

In [ ]:
# Download Tiny-ImageNet-200 to /tmp (EPHEMERAL: keeps your saved Version small --
# only checkpoints go to /kaggle/working). Re-downloads each session (~2-3 min).
# Faster alternative: attach a Kaggle tiny-imagenet dataset and set DATA_DIR to it,
# then comment this cell out. Requires Internet ON.
import os, zipfile, glob, shutil
DATA_DIR = "/tmp/tiny-imagenet-200"
ZIP      = "/tmp/tiny-imagenet-200.zip"
SRC      = "http://cs231n.stanford.edu/tiny-imagenet-200.zip"

if not os.path.isdir(os.path.join(DATA_DIR, "train")):
    if not os.path.exists(ZIP):
        print("downloading tiny-imagenet-200 (~240MB)...")
        os.system(f"wget -q -O {ZIP} {SRC}")
    print("extracting...")
    with zipfile.ZipFile(ZIP) as z:
        z.extractall("/tmp")

# TRAIN: train/<cls>/images/*.JPEG -> train/<cls>/*.JPEG  (ImageFolder layout)
tr = os.path.join(DATA_DIR, "train")
for cls in os.listdir(tr):
    sub = os.path.join(tr, cls, "images")
    if os.path.isdir(sub):
        for f in glob.glob(os.path.join(sub, "*.JPEG")):
            shutil.move(f, os.path.join(tr, cls))
        shutil.rmtree(sub)
    for b in glob.glob(os.path.join(tr, cls, "*_boxes.txt")):
        os.remove(b)

# VAL: flat val/images + val_annotations.txt -> val/<cls>/*.JPEG  (evaluate.py reads val/)
val = os.path.join(DATA_DIR, "val")
ann = os.path.join(val, "val_annotations.txt")
if os.path.exists(ann):
    with open(ann) as f:
        for line in f:
            img, cls = line.split("\t")[:2]
            os.makedirs(os.path.join(val, cls), exist_ok=True)
            s = os.path.join(val, "images", img)
            if os.path.exists(s):
                shutil.move(s, os.path.join(val, cls, img))
    shutil.rmtree(os.path.join(val, "images"), ignore_errors=True)
    os.remove(ann)

print("train classes:", len(glob.glob(tr+"/*")),
      "| val classes:", len(glob.glob(val+"/*")), "| DATA_DIR:", DATA_DIR)

## 5. Settings  *(edits the cloned config so train & eval agree)*

In [ ]:
# ── Settings for THIS notebook ─────────────────────────────────────────
EPOCHS     = 100          # full target; resume across sessions until reached
SAVE_FREQ  = 1            # overwrite checkpoint-last.pt EVERY epoch (12h safety)

# Fallback if the download cell was commented out (attached-dataset workflow):
try:
    DATA_DIR
except NameError:
    DATA_DIR = "/kaggle/input/tiny-imagenet/tiny-imagenet-200"  # <-- your attached dataset
    print("DATA_DIR not set by a download cell; using:", DATA_DIR)

CONFIG  = "configs/tiny_imagenet/jit-s2-vmamba-dstate1.yaml"
RUN     = "exp_vmamba_dstate1"
OUT_DIR = f"/kaggle/working/{RUN}"   # persists in the saved Version

# Sync the cloned config so BOTH training and evaluation read identical values
# (evaluate.py has no model CLI overrides -- it builds the model straight from
# config). Every conditioning arm stays OFF (adaLN-Zero baseline scaffold).
import re
def set_cfg(path, kv):
    s = open(path).read()
    for k, v in kv.items():
        s, n = re.subn(rf"(?m)^(\s*{k}:).*$", rf"\1 {v}", s, count=1)
        assert n == 1, f"key {k!r} not found in {path}"
    open(path, "w").write(s)
set_cfg(CONFIG, {"data_dir": DATA_DIR, "d_state": 1, "expand": 1, "K": 4,
                 "in_context_len": 0, "state_init": "none", "ssc": "none",
                 "adaln_cond": "full", "ffn": "swiglu"})

# ── Resume across 12h sessions ─────────────────────────────────────
# 1st run: RESUME_INPUT = None. Next session: Save Version, attach THIS notebook's
# previous output as an input, set RESUME_INPUT to ".../exp_vmamba_dstate1".
RESUME_INPUT = None
import os
RESUME_CKPT = None
for c in ([os.path.join(RESUME_INPUT, "checkpoint-last.pt")] if RESUME_INPUT else []) + \
         [os.path.join(OUT_DIR, "checkpoint-last.pt")]:
    if c and os.path.exists(c):
        RESUME_CKPT = c; break
print("config :", CONFIG, "|", "vmamba d_state=1 expand=1 | arms: all off (adaLN-Zero baseline)")
print("data   :", DATA_DIR, "(exists:", os.path.isdir(DATA_DIR), ")")
print("out    :", OUT_DIR, "| resume:", RESUME_CKPT or "(fresh)")

## 6. Train  *(Save Version before 12h; set RESUME_INPUT next session)*

In [ ]:
# ── TRAIN ──────────────────────────────────────────────────────────────
# data_dir + the model knobs now live in the (edited) config; we only override run-specific
# knobs here. Runs in a subprocess (fresh torch, no restart).
CMD = (f"python scripts/run_experiment.py --config {CONFIG} "
       f"checkpoint.output_dir={OUT_DIR} checkpoint.save_last_freq={SAVE_FREQ} "
       f"training.epochs={EPOCHS}")
if RESUME_CKPT:
    CMD += f" checkpoint.resume_from={RESUME_CKPT}"
print(CMD, "\n" + "="*70)
!{CMD}

## 7. Evaluate  *(comment out the whole cell to skip)*

In [ ]:
import glob, os

# Find the checkpoint to evaluate: this session's OUT_DIR first, then any
# attached notebook output (Save Version -> attach it as an input). Searches
# instead of hardcoding a notebook hash, so nothing needs editing after the
# first Save Version.
cands = []
for root in (os.path.dirname(OUT_DIR), "/kaggle/input"):
    for which in ("checkpoint-best.pt", "checkpoint-last.pt"):
        cands += sorted(glob.glob(f"{root}/**/{RUN}/{which}", recursive=True))
assert cands, f"no checkpoint found for {RUN} -- train first, or attach the saved Version"
EVAL_CKPT = cands[0]
EVAL_OUT  = f"/kaggle/working/{RUN}/eval"
print("eval ckpt:", EVAL_CKPT)

EVAL_CMD = (f"python scripts/evaluate.py --config {CONFIG} --checkpoint {EVAL_CKPT} "
            f"--n_samples 10000 --batch_size 200 --ema 1 "
            f"--cfg_scale 2.5 --cfg_interval 0.1 1.0 --out_dir {EVAL_OUT}")
print(EVAL_CMD, "\n" + "="*70)
!PYTHONPATH=. {EVAL_CMD}

## Notes
- Checkpoints (`-last`/`-best`/`-ep###`) with both EMA copies + optimizer are in `OUT_DIR` under `/kaggle/working`, so a resumed run equals a continuous one.
- Keep `EPOCHS`, `cfg_scale`, `DATA_DIR`, seed and sampler identical to the baseline and the two Spatial runs, so the only variable is the mixer.
- Comparisons: **this vs the d_state-16 baseline** = the effect of the SSM state size alone (the VMamba paper never isolated it). **Spatial no-gate vs this** = 4 scans + CrossMerge → 1 scan + SASF (+ the Spatial block's LPU).
- −0.83M params vs the baseline (31.03M → 30.20M): only `x_proj` (fewer B/C rows) and `A_logs` shrink.
- Sister notebooks: `jit-s2-spatial-tinyin-gate.ipynb`, `jit-s2-spatial-tinyin-nogate.ipynb`. Their verify cell prints the it/s of all three models side by side.
- Data lives in `/tmp` (not saved in the Version) to keep commits small; it re-downloads each session. To skip that, attach a Kaggle tiny-imagenet dataset and set `DATA_DIR` to it.